# Colour palettes

**Three kinds of colour, and picking the right one.**

The single most useful idea in this whole folder:

```text
CATEGORICAL  -- unordered things (regions, species). Distinct hues.
SEQUENTIAL   -- a quantity from low to high. One hue, light to dark.
DIVERGING    -- distance from a meaningful middle (zero, average).
                Two hues meeting at a neutral centre.
```

Use the wrong family and the picture lies before anyone reads a number.

**What it shows:**

- the three families, side by side, on data that suits each
- a sequential palette used on categories -- inventing an order
- a diverging palette without a meaningful centre -- inventing a middle
- how many categories is too many (about seven)

---

*Chapter:* `color` — palettes, colour blindness, and why rainbow lies  
*Run the cells in order.* Every figure is also written to `viz/output/color/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np

from vizkit import save

# Where save() files this lesson's output: viz/output/color/
LESSON = "color/palettes"


## One shared random generator

Seeded, so the four figures below look the same on every machine.


In [ ]:
rng = np.random.default_rng(5)


## 1. The three families

Three strips, three jobs. Read them as: distinct hues for unordered things, one hue getting darker for a quantity, two hues meeting at a neutral middle for distance from a centre.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5))

gradient = np.linspace(0, 1, 256).reshape(1, -1)
for ax, cmap, label in [
    (axes[0], "tab10", "CATEGORICAL (tab10) -- unordered groups"),
    (axes[1], "viridis", "SEQUENTIAL (viridis) -- low to high"),
    (axes[2], "RdBu_r", "DIVERGING (RdBu_r) -- below / at / above a centre"),
]:
    ax.imshow(gradient, aspect="auto", cmap=cmap)
    ax.set_title(label, fontsize=10, loc="left")
    ax.set_xticks([]); ax.set_yticks([])

fig.tight_layout()
save(fig, LESSON, "three-families");


## 2. Sequential used on categories = an order that is not there

Nothing about apple/banana/cherry is ordered, but the viridis ramp says otherwise — the reader gets an ordering for free that the data never had.


In [ ]:
fruit = ["apple", "banana", "cherry", "date", "elderberry"]
counts = [23, 17, 35, 12, 28]

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))

wrong = plt.get_cmap("viridis")(np.linspace(0.15, 0.9, len(fruit)))
left.bar(fruit, counts, color=wrong)
left.set_title("Sequential on categories:\nimplies apple < banana < cherry", fontsize=10)

right.bar(fruit, counts, color=plt.get_cmap("tab10").colors[:len(fruit)])
right.set_title("Categorical: no order implied", fontsize=10)

fig.tight_layout()
save(fig, LESSON, "sequential-on-categories");


## 3. Diverging needs a real centre

A diverging map's white is a claim: 'this is the middle'. On plain positive data there is no middle, so white lands wherever the data happened to average. When there *is* one, pin it with `vmin`/`vmax`.


In [ ]:
grid_positive = rng.uniform(10, 90, (12, 12))                 # no natural middle
grid_change = rng.normal(0, 25, (12, 12))                     # centred on zero

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

im0 = axes[0].imshow(grid_positive, cmap="RdBu_r")
axes[0].set_title("Diverging on 10-90:\nwhite is meaningless here", fontsize=10)
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(grid_positive, cmap="viridis")
axes[1].set_title("Sequential: correct for\na plain quantity", fontsize=10)
fig.colorbar(im1, ax=axes[1])

# For diverging, pin the centre yourself -- otherwise it lands wherever the
# data happens to average out.
limit = np.abs(grid_change).max()
im2 = axes[2].imshow(grid_change, cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[2].set_title("Diverging on change:\nwhite = zero, and vmin/vmax pinned", fontsize=10)
fig.colorbar(im2, ax=axes[2])

fig.tight_layout()
save(fig, LESSON, "diverging-centre");


## 4. Too many categories

Past about seven categories, matching line to legend stops being possible. The fix is not a bigger palette — it is deciding which two series you are actually talking about.


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

x = np.arange(20)
for i in range(14):
    left.plot(x, np.cumsum(rng.normal(0, 1, 20)), label=f"item {i}")
left.legend(fontsize=6, ncol=2)
left.set_title("14 colours: nobody can match line to legend", fontsize=10)

series = [np.cumsum(rng.normal(0, 1, 20)) for _ in range(14)]
for values in series:
    right.plot(x, values, color="#CCCCCC", lw=1)
for i in (2, 7):
    right.plot(x, series[i], lw=2.5, label=f"item {i}")
right.legend(fontsize=9)
right.set_title("Grey the rest, colour the two you mean", fontsize=10)

fig.tight_layout()
save(fig, LESSON, "too-many-categories");


## Rules of thumb

```text
Pick the family from the data, not from taste:
  unordered groups        -> categorical (tab10), max ~7
  a quantity low to high  -> sequential (viridis)
  distance from a centre  -> diverging (RdBu_r), and PIN vmin/vmax
More than 7 groups? Colour the ones you are talking about, grey the rest.
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Swap `viridis` for `plasma` and `RdBu_r` for `PiYG` in section 1. Do the three families still read as three families?
2. In section 3, drop the `vmin`/`vmax` from the third panel. Where does white land now, and what would a reader conclude from it?
3. In section 4, colour three series instead of two. At what number does the greying trick stop working?


In [ ]:
# your turn


---

**Previous:** [`foundations/saving`](../foundations/saving.ipynb)  
**Next:** [`color/colorblind`](colorblind.ipynb)
